## Реализации сервиса с 3 хранилищами

Имеется небольшое приложение (его заготовка в папке starter), которое должно собирать карточку аккаунта из нескольких источников данных.

В этом задании нужно поработать сразу с 3 хранилищами:
- `PostgreSQL` — хранит сами аккаунты
- `MongoDB` — хранит журнал событий по аккаунту
- `Redis` (или любое другое `k-v` хранилище, например Valkey) — хранит временный код подтверждения с TTL

Требуется реализовать 2 версии сервиса:
- синхронную: `AccountCardService`
- асинхронную: `AsyncAccountCardService`

Также требуется реализовать хранилища для обеих версий.

### Что должен уметь сервис

1. Создавать аккаунт `create_account(email)`
- аккаунт должен сохраняться в `PostgreSQL`
- в `MongoDB` должно записываться событие `account_created`

2. Создавать временный код подтверждения `set_verification_code(account_id, ttl_seconds=300)`
- код должен генерироваться в приложении
- код должен сохраняться в `Redis` с TTL
- в `MongoDB` должно записываться событие `verification_code_set`

3. Собирать карточку аккаунта `get_account_card(account_id)`
- нужно получить аккаунт из `PostgreSQL`
- проверить, есть ли активный код в `Redis`
- получить последние события из `MongoDB`
- вернуть всё это в виде одной структуры `AccountCard`

### Что именно нужно реализовать

В каркасе (`starter/`) задания я разметил практически всё что нужно реализовать меткой #TODO

Рекомендуется сначала сделать синхонный вариант. Пояснения есть в `starter/auth/service_sync.py`

#### Синхронный вариант
- `PostgresAccountsRepository`
- `MongoAuditRepository`
- `RedisCodeRepository`
- `AccountCardService`

#### Асинхронный вариант
- `AsyncPostgresAccountsRepository`
- `AsyncMongoAuditRepository`
- `AsyncRedisCodeRepository`
- `AsyncAccountCardService`

+ модифицировать код app_async.py так чтобы он использовал асинхронность

### Что дано

В папке `starter/` находится каркас задания.

Там уже есть:
- модели
- протоколы
- конфиг
- точки входа `app_sync.py` и `app_async.py`
- пустые классы репозиториев и сервисов

### Как проверить решение

Для обоих вариантов есть минимальные сценарии:
- `starter/app_sync.py`
- `starter/app_async.py`

После реализации кода они должны корректно отрабатывать. Данные должны появиться в базах, для визуализации можно воспользоваться DbGate

### Технические ограничения

- не использовать ORM (ORM в индустрии практически не используется)
- нужно работать через драйверы БД напрямую
- соблюдать сигнатуры, заданные в `protocols.py`
- не усложнять решение лишней архитектурой

Для ознакомления с API Баз данных, смотри документацию (и материал лекции):

    https://www.psycopg.org/docs/
    https://redis.readthedocs.io/en/latest/
    https://pymongo.readthedocs.io/en/stable/
    
Сами базы данных можно установить локально при помощи Docker
- https://hub.docker.com/_/postgres
- https://hub.docker.com/_/mongo
- https://hub.docker.com/_/redis

## Асинхронное взаимодействие

Есть модуль `hident.py` (приложен к домашнему заданию), который содержит логику обработки криптографических хешей. Функция `identify_hashes(input_hash)` позволяет узнать алгоритм хеширования по виду самого хеша, корутина `long_solve_hash(input_hash, alg)` имитирует работу 'обратного преобразования' хеша ([как это?](https://hashcat.net/hashcat/)), то есть позволяет восстановить секретное слово
 
 Требуется на [aiohttp](https://docs.aiohttp.org/en/stable/web.html) разработать приложение (сервер), которое будет иметь 3 ручки:
 - для вычисления возможного алгоритма хеширования по хешу `host:port/define/{hash}`
 - 2 ручки для асинхронного вычисления секретного слова (нужен именно асинхронный вариант, так как long_solve_hash работает очень долго - представьте надоедливое вращающееся колёсико загрузки, которое было бы в синхронном варианте)
     - `host:port/createSolveTask?hash={hash}&algorithm={algorithm}`
     - `host:port/getPassword?taskId={tid}`
 
 Напишите клиента, который будет использовать сервер:
 - отправлять 1 любой хеш, например `c4ca4238a0b923820dcc509a6f75849b`, получать его возможные алгоритмы 
 - по каждому алгоритму вычислять секретное слово
 
 Полностью контракт сервиса может выглядеть так:
 ```go
rpc createSolveTask (CreateSolveTaskIn) CreateSolveTaskOut `Создать задачу на преобразование хеша`

message CreateSolveTaskIn  {inputHash string, alg string}
message CreateSolveTaskOut {taskId int}

rpc getPassword (GetPasswordIn) GetPasswordOut `Получить результат преобразования`

message GetPasswordIn      {taskId int}
message GetPasswordOut     {password *string, err *Error}

rpc define (DefineIn) DefineOut `Определить алгоритм`

message DefineIn      {hash string}
message DefineOut     {algs []string}

message Error              {code int, message string}  `допустимые значения в ErrCode, ErrMessage`
const ErrCode {
    ERR_TASK_NOT_FOUND          = 1000
    ERR_TASK_NOT_FINISHED       = 1001
}
const ErrMessage {
    ERR_TASK_NOT_FOUND          = "task not found"
    ERR_TASK_NOT_FINISHED       = "task not finished"
}
```